
## Cropping Roberts images by hand

The code below automatically takes one through all the scraped Julia Roberts images and enables one to crop them so that they are in such a shape
that they can be added to the CelebA dataset without disrupting the ordinary learning process much (if one adds them in the original form then the model will switch to a mixture of learning to generate CelebA images and learning to generate more general images where the person is on the image with full body and not in the center). When you get shown a picture, you need to click two points and then the rectangle that is largest and fits within these two points will be the crop saved into the folder. If the images is not suitable for cropping you can also skip. Either way, the image will be deleted in the original folder. 




In [ ]:
import cv2
import os
import glob

# === CONFIGURATION ===
input_folder = 'roberts_images'       # Folder with original images
output_folder = 'croppedbyhand'     # Folder for saving cropped images
os.makedirs(output_folder, exist_ok=True)

# CelebA image dimensions
CELEBA_WIDTH = 178
CELEBA_HEIGHT = 218
CELEBA_ASPECT = CELEBA_HEIGHT / CELEBA_WIDTH
DISPLAY_MAX_WIDTH = 700

# Store points clicked in display space
points_display = []

def mouse_callback(event, x, y, flags, param):
    global points_display
    if event == cv2.EVENT_LBUTTONDOWN and len(points_display) < 2:
        points_display.append((x, y))
        print(f"🖱️  Point {len(points_display)}: ({x}, {y})")

def crop_with_aspect_ratio(img, pt1, pt2):
    x1, y1 = pt1
    x2, y2 = pt2
    x_min, x_max = sorted([x1, x2])
    y_min, y_max = sorted([y1, y2])

    crop_w = x_max - x_min
    crop_h = y_max - y_min
    box_aspect = crop_h / crop_w

    if box_aspect > CELEBA_ASPECT:
        # Too tall → crop height
        new_h = int(crop_w * CELEBA_ASPECT)
        y_center = (y_min + y_max) // 2
        y_min_crop = max(0, y_center - new_h // 2)
        y_max_crop = min(img.shape[0], y_min_crop + new_h)
        x_min_crop, x_max_crop = x_min, x_max
    else:
        # Too wide → crop width
        new_w = int(crop_h / CELEBA_ASPECT)
        x_center = (x_min + x_max) // 2
        x_min_crop = max(0, x_center - new_w // 2)
        x_max_crop = min(img.shape[1], x_min_crop + new_w)
        y_min_crop, y_max_crop = y_min, y_max

    cropped = img[y_min_crop:y_max_crop, x_min_crop:x_max_crop]
    resized = cv2.resize(cropped, (CELEBA_WIDTH, CELEBA_HEIGHT))
    return resized

# === MAIN LOOP ===
image_paths = sorted(glob.glob(os.path.join(input_folder, '*.*')))

for image_path in image_paths:
    img = cv2.imread(image_path)
    if img is None:
        print(f"❌ Could not read {image_path}")
        continue

    original_h, original_w = img.shape[:2]
    scale_factor = min(DISPLAY_MAX_WIDTH / original_w, 1.0)
    display_img = cv2.resize(img, (int(original_w * scale_factor), int(original_h * scale_factor)))

    points_display = []
    skip_image = False

    # Display window and mouse handling
    window_name = "Click 2 corners | 's'=skip, 'q'=quit"
    cv2.namedWindow(window_name)
    cv2.setMouseCallback(window_name, mouse_callback)

    while True:
        temp_img = display_img.copy()
        for pt in points_display:
            cv2.circle(temp_img, pt, 4, (0, 255, 0), -1)
        cv2.imshow(window_name, temp_img)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            print("👋 Quitting...")
            cv2.destroyAllWindows()
            exit()
        elif key == ord('s'):
            print(f"⏭️ Skipped: {image_path}")
            skip_image = True
            break
        elif len(points_display) == 2:
            break

    cv2.destroyAllWindows()

    if skip_image:
        os.remove(image_path)
        print(f"🗑️ Deleted (skipped): {image_path}")
        continue

    # Map display clicks to original image coordinates
    points_original = [(int(x / scale_factor), int(y / scale_factor)) for (x, y) in points_display]

    # Crop with fixed aspect and resize
    output_img = crop_with_aspect_ratio(img, points_original[0], points_original[1])

    # Save cropped image
    filename = os.path.basename(image_path)
    save_path = os.path.join(output_folder, filename)
    cv2.imwrite(save_path, output_img)
    print(f"✅ Saved: {save_path}")

    # Delete original
    os.remove(image_path)
    print(f"🗑️ Deleted original: {image_path}")

print("🎉 All images processed.")
